### Key Design Clarifications Before Code Execution

Before running the blocks, note the **4 architectural decisions** made in the code below to ensure everything works smoothly with `sarvamai/sarvam-2b-v0.5`:

1. **Data Merging Across 4 Languages (Block 5 & 9):** The original code only loads and merges 2 CSV files (`input_lang` and `target_lang`) against `word_original` (`en`). Since we also need Hindi (`hi`) in `principal_lang_candidates`, the updated code loads `hi/clean.csv`, `input_lang/clean.csv`, and `target_lang/clean.csv` and merges all of them on `word_original` so every row has `"en"`, `"hi"`, `input_lang`, and `target_lang` columns.


2. **Intersection Filtering Rule (Block 11):** The original paper filters out rows where the latent token IDs overlap with the target token IDs (`intersection = set(out_token_id).intersection(set(latent_token_id))`) so shared substrings don't artificially inflate latent probabilities. In your 4-candidate setup, `target_lang` is itself in the candidate list (where intersection is 100%). The code below applies the disjoint check **only** when $c \neq \text{target\_lang}$, and **skips** the strict disjoint check between `hi` and Marathi (`mar`) if they share identical Devanagari root prefixes—otherwise, cognate overlap would drop most of your dataset. (You can toggle `strict_disjoint = False` in Block 11).


3. **Plotting 4 Subplots vs. `plot_ci_plus_heatmap` (Block 17):** In `utils.py`, `plot_ci_plus_heatmap` creates its own standalone `Figure` internally rather than accepting a subplot `ax`. To give you **Option A** cleanly without breaking `utils.py`, Block 17 provides both:


* A $2 \times 2$ unified Matplotlib grid using `plot_ci` (which accepts `ax`) with an entropy bar above each subplot.
* A loop calling the original `plot_ci_plus_heatmap` 4 times to save the exact paper-style PDFs for each candidate language.




4. **Sarvam-2B Architecture:** `sarvamai/sarvam-2b-v0.5` uses the `LlamaForCausalLM` architecture (`28` layers, `2048` hidden size) and a SentencePiece tokenizer (`▁` space prefix), so `LlamaHelper` and `llama.model.model.norm` work out of the box.

In [1]:
# import os

# selected_GPU = "1"
# selected_GPU_int = int(selected_GPU)
# os.environ["CUDA_VISIBLE_DEVICES"] = selected_GPU  # Physical GPU to select

# import torch

# # Create a small tensor so this process uses the selected GPU
# x = torch.empty(1, device="cuda")
# torch.cuda.synchronize()

# # free, total = torch.cuda.mem_get_info(0)

# # print("Kernel PID:", os.getpid())
# print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
# # print("Visible GPU count:", torch.cuda.device_count())
# # print("GPU name:", torch.cuda.get_device_name(1))
# # print(f"Free VRAM: {free / 1024**3:.2f} GiB")

In [2]:
!nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv

index, memory.used [MiB], memory.free [MiB]
0, 18 MiB, 32242 MiB
1, 4454 MiB, 41034 MiB


#### Block 1: Autoreload & Imports

Loads standard libraries and `LlamaHelper` / `utils` from the repository.

In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sys
import os
# Add the parent directory to sys.path for llamawrapper.py is located there
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
from dataclasses import dataclass
import json
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from llamawrapper import load_unemb_only, LlamaHelper
from scipy.stats import bootstrap
from utils import plot_ci, plot_ci_plus_heatmap
from tqdm import tqdm

# fix random seed
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

/user1/anish/.conda/envs/btp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Block 2: Parameters & Candidate Language Setup

Sets the regional translation pair (`input_lang` $\rightarrow$ `target_lang`), points `custom_model` to `sarvamai/sarvam-2b-v0.5`, and defines your 4 candidate languages.

In [4]:
input_lang = 'ben'   # e.g., 'ben', 'mar', 'tel', 'tam'
target_lang = 'pun'  # e.g., 'ben', 'mar', 'tel', 'tam'
model_size = '2b'
# custom_model = 'sarvamai/sarvam-2b-v0.5'
# custom_model = 'sarvamai/sarvam-30b'
custom_model = 'google/gemma-7b'
single_token_only = False
multi_token_only = False
out_dir = './visuals'
hf_token = None # Or your HF token string if needed

# Principal language candidates to investigate
principal_lang_candidates = ["en", "hi", input_lang, target_lang]
# Preserve order while deduplicating in case input/target is 'en' or 'hi'
candidates = list(dict.fromkeys(principal_lang_candidates))
print("Investigating candidates:", candidates)

Investigating candidates: ['en', 'hi', 'ben', 'pun']


#### Block 3: Token Filter Validation

Ensures mutually exclusive token filtering flags.

In [5]:
if single_token_only and multi_token_only:
    raise ValueError('single_token_only and multi_token_only cannot be True at the same time')

#### Block 4: Load Model & Unembedding Layer

Loads `sarvamai/sarvam-2b-v0.5` via `LlamaHelper` and extracts the RMSNorm + `lm_head` sequential unembedding projection. (Note: Block 4 from the original file `load_unemb_only(model_size)` is skipped because it hardcodes Llama-2 local paths and is immediately overwritten in Block 7 anyway).

In [6]:
import torch
import os

print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)}"
    )
    print(
        f"Free memory: {torch.cuda.mem_get_info(i)[0] / 1024**3:.2f} GB"
    )

CUDA available: True
Visible GPU count: 2
CUDA_VISIBLE_DEVICES: None
GPU 0: NVIDIA L40S
Free memory: 39.65 GB
GPU 1: NVIDIA RTX 5000 Ada Generation
Free memory: 31.18 GB


In [7]:
llama = LlamaHelper(
    dir=custom_model, 
    load_in_8bit=True,
    hf_token=hf_token,
#     device_map="cuda:0"
)
tokenizer = llama.tokenizer
model = llama.model

unemb = nn.Sequential(llama.model.model.norm, llama.model.lm_head)
print(unemb)

# Check device placement of the unembedding layer
print("lm_head device:", next(llama.model.lm_head.parameters()).device)
print("Model device map / primary device:", getattr(llama.model, "hf_device_map", "No device map"))

# Prepare for energy plots
U = list(unemb[1].parameters())[0].detach().cpu().float()
weights = list(unemb[0].parameters())[0].detach().cpu().float()
print(f'U {U.shape} weights {weights.unsqueeze(0).shape}')
U_weighted = U.clone() 
U_weighted *= weights.unsqueeze(0)
U_normalized = U_weighted / ((U_weighted**2).sum(dim=1, keepdim=True))**0.5
v = U.shape[0]
TT = U_normalized.T @ U_normalized
avgUU = (((U_normalized.T @ U_normalized)**2).sum() / v**2)**0.5
print("avgUU:", avgUU.item())

[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.
Loading weights: 100%|███████████████████████████████████████████████| 254/254 [00:03<00:00, 76.26it/s]


Sequential(
  (0): GemmaRMSNorm((3072,), eps=1e-06)
  (1): Linear(in_features=3072, out_features=256000, bias=False)
)
lm_head device: cuda:0
Model device map / primary device: {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1}
U torch.Size([256000, 3072]) weights torch.Size([1, 3072])
avgUU: 0.8637301325798035


In [8]:
# U = list(unemb[1].parameters())[0].detach().cpu().float()

#### Block 5: Load & Merge Multi-Language DataFrames

Loads the CSV files for `"hi"`, `input_lang`, and `target_lang`, and merges them on `word_original` (`"en"`) so all 4 candidate columns exist in one DataFrame.

In [9]:
prefix = "./data/langs/"

# Load target language first to apply single/multi token filtering if enabled
df_tgt = pd.read_csv(f'{prefix}{target_lang}/clean.csv').reindex()

count = 0
for idx, word in enumerate(df_tgt['word_translation']):
    word_str = str(word)
    # print(word_str)
    if word_str in tokenizer.get_vocab() or '▁' + word_str in tokenizer.get_vocab():
        count += 1
        if multi_token_only:
            df_tgt.drop(idx, inplace=True)
    elif single_token_only:
        df_tgt.drop(idx, inplace=True)

print(f'for {target_lang} {count} of {len(df_tgt)} are single tokens')

# Build merged dataframe across all non-English candidate languages
df_merged = df_tgt.rename(columns={'word_original': 'en', 'word_translation': target_lang})

for lang in candidates:
    if lang in ['en', target_lang]:
        continue
    df_lang = pd.read_csv(f'{prefix}{lang}/clean.csv').reindex()
    df_lang = df_lang.rename(columns={'word_original': 'en', 'word_translation': lang})
    # Keep only 'en' and the language column to avoid extra column collisions
    df_merged = df_merged.merge(df_lang[['en', lang]], on=['en'])

# Drop rows where English word is literally a substring of the target word
if target_lang != 'en':
    for i, row in df_merged.iterrows():
        if str(row['en']).lower() in str(row[target_lang]).lower():
            df_merged.drop(i, inplace=True)

df_merged = df_merged.reset_index(drop=True)
print(f'final length of df_merged: {len(df_merged)}')
df_merged.head()

for pun 0 of 132 are single tokens
final length of df_merged: 132


,en,pun,hi,ben
0,book,ਕਿਤਾਬ,किताब,বই
1,cloud,ਬੱਦਲ,बादल,মেঘ
2,bag,ਥੈਲਾ,थैला,ব্যাগ
3,mouth,ਮੂੰਹ,मुँह,মুখ
4,soil,ਮਿੱਟੀ,मिट्टी,মাটি


In [10]:
print(len(df_merged))

132


#### Block 6: Token Prefix & Vocabulary Lookup Helpers

Collects all valid token IDs corresponding to prefixes of a target word (both with and without the SentencePiece leading space `▁`).

In [11]:
def token_prefixes(token_str: str):
    token_str = str(token_str).strip()
    n = len(token_str)
    tokens = [token_str[:i] for i in range(1, n+1)]
    return tokens 

def add_spaces(tokens):
    return ['▁' + t for t in tokens] + tokens

def capitalizations(tokens):
    return list(set(tokens))

def unicode_prefix_tokid(zh_char="云", tokenizer=tokenizer):
    try:
        start = zh_char.encode().__str__()[2:-1].split('\\x')[1]
        unicode_format = '<0x%s>'
        start_key = unicode_format % start.upper()
        if start_key in tokenizer.get_vocab():
            return tokenizer.get_vocab()[start_key]
    except Exception:
        pass
    return None

def process_tokens(token_str: str, tokenizer, lang):
    with_prefixes = token_prefixes(token_str)
    with_spaces = add_spaces(with_prefixes)
    with_capitalizations = capitalizations(with_spaces)
    final_tokens = []
    vocab = tokenizer.get_vocab()
    for tok in with_capitalizations:
        if tok in vocab:
            final_tokens.append(vocab[tok])
    # Fallback to first token from tokenizer.encode if prefix matching misses Indic ligatures
    if len(final_tokens) == 0:
        encoded = tokenizer.encode(str(token_str).strip(), add_special_tokens=False)
        if len(encoded) > 0:
            final_tokens.append(encoded[0])
    return list(set(final_tokens))

#### Block 7: Few-Shot Prompt Generator & Multi-Candidate Token Extractor

Constructs the $k$-shot translation prompt (`lang1` $\rightarrow$ `lang2`) and extracts token ID lists for all 4 languages in `candidates`.

In [12]:
id2voc = {id: voc for voc, id in llama.tokenizer.get_vocab().items()}

def get_tokens(token_ids, id2voc=id2voc):
    return [id2voc[tokid] for tokid in token_ids]

def compute_entropy(probas):
    return (-probas * torch.log2(probas + 1e-12)).sum(dim=-1)

# Mapped names for English, Hindi, and Indian regional languages
lang2name = {
    'en': 'English',
    'hi': 'हिन्दी',
    'ben': 'বাংলা',
    'mar': 'मराठी',
    'tel': 'తెలుగు',
    'tam': 'தமிழ்',
    'guj': 'ગુજરાતી',
    'kan': 'ಕನ್ನಡ',
    'mal': 'മലയാളം',
    'pun': 'ਪੰਜਾਬੀ'
}
lang2english_name = {
    'en': 'English',
    'hi': 'Hindi',
    'ben': 'Bengali',
    'mar': 'Marathi',
    'tel': 'Telugu',
    'tam': 'Tamil',
    'guj': 'Gujarati',
    'pun': 'Punjabi'
}

# Seems like the sarvam model doesn't understand that it has to perform translation (because the tam probabilites were close to 0 when doing hi -> tam),
# modifying the prompt to make it more clear by doing the following:
# 1. increase k, 
# 2. Add a header that says that the model has to translate, ---> will inspect 
# 3. Remove the double quotes to reduce confusion ---> This led to model doing nothing, so we reverted back to using quotes 

def sample(df, ind, k=10, tokenizer=tokenizer, lang1='ben', lang2='tam', candidates=candidates, strict_disjoint=False):
    df = df.reset_index(drop=True)
    temp = df[df.index != ind]
    sample_df = pd.concat([temp.sample(k-1), df[df.index == ind]], axis=0)
    
    prompt = f"Translate the following words from {lang2english_name[lang1]} to {lang2english_name[lang2]}.\n\n"
    # prompt = ""
    for idx, (df_idx, row) in enumerate(sample_df.iterrows()):
        if idx < k-1:
            prompt += f'{lang2name[lang1]}: "{row[lang1]}" - {lang2name[lang2]}: "{row[lang2]}"\n'
            # prompt += f"{lang2name[lang1]}: {row[lang1]} - {lang2name[lang2]}: "
        else:
            prompt += f'{lang2name[lang1]}: "{row[lang1]}" - {lang2name[lang2]}: "'
            
            item = {
                'prompt': prompt,
                'in_token_str': row[lang1],
                'out_token_str': row[lang2],
            }
            
            out_token_id = process_tokens(row[lang2], tokenizer, lang2)
            if len(out_token_id) == 0:
                yield None
                return
            item['out_token_id'] = out_token_id
            
            # Process all 4 candidate languages
            for c in candidates:
                c_str = str(row[c])
                c_tok_id = process_tokens(c_str, tokenizer, c)
                if len(c_tok_id) == 0:
                    yield None
                    return
                
                # Optional disjoint filter against target language
                if strict_disjoint and c != lang2:
                    if len(set(out_token_id).intersection(set(c_tok_id))) > 0:
                        yield None
                        return
                        
                item[f'{c}_token_str'] = c_str
                item[f'{c}_token_id'] = c_tok_id
                
            yield item

#### Block 8: Build & Save Dataset

Iterates over `df_merged` to build the few-shot prompt dataset and saves it to CSV.

In [13]:
dataset = []
for ind in tqdm(range(len(df_merged))):
    d = next(sample(df_merged, ind, lang1=input_lang, lang2=target_lang, candidates=candidates))
    if d is None:
        continue
    dataset.append(d)

df = pd.DataFrame(dataset)
save_path = f'{os.path.join(out_dir, custom_model)}/translation'
os.makedirs(save_path, exist_ok=True)

if single_token_only:
    df.to_csv(f'{save_path}/{model_size}_{input_lang}_{target_lang}_dataset_single_token.csv', index=False)
elif multi_token_only:
    df.to_csv(f'{save_path}/{model_size}_{input_lang}_{target_lang}_dataset_multi_token.csv', index=False)
else:
    df.to_csv(f'{save_path}/{model_size}_{input_lang}_{target_lang}_dataset.csv', index=False)

print(f"Valid samples generated: {len(df)}")
df.head()

100%|████████████████████████████████████████████████████████████████| 132/132 [01:25<00:00,  1.55it/s]

Valid samples generated: 132


,prompt,in_token_str,out_token_str,out_token_id,en_token_str,en_token_id,hi_token_str,hi_token_id,ben_token_str,ben_token_id,pun_token_str,pun_token_id
0,Translate the following words from Bengali to ...,বই,ਕਿਤਾਬ,"[44375, 163774, 237582]",book,"[235268, 17862, 518, 1036, 2545, 2870, 31931, ...",किताब,"[12704, 235456, 49122, 2280, 151881, 219113]",বই,"[27706, 236571]",ਕਿਤਾਬ,"[44375, 163774, 237582]"
1,Translate the following words from Bengali to ...,মেঘ,ਬੱਦਲ,"[238960, 89771]",cloud,"[7008, 44193, 2059, 12238, 57137, 786, 14418, ...",बादल,"[112901, 63081, 235825, 7320, 94046]",মেঘ,"[39808, 236821]",ਬੱਦਲ,"[238960, 89771]"
2,Translate the following words from Bengali to ...,ব্যাগ,ਥੈਲਾ,[241670],bag,"[235268, 518, 6214, 4079, 2070, 4221]",थैला,"[35193, 236277]",ব্যাগ,"[236571, 27706, 187547]",ਥੈਲਾ,[241670]
3,Translate the following words from Bengali to ...,মুখ,ਮੂੰਹ,"[238307, 75614]",mouth,"[705, 519, 17982, 120446, 40651, 9615, 17848, ...",मुँह,"[39728, 4745, 235579, 84372]",মুখ,"[39808, 236821]",ਮੂੰਹ,"[238307, 75614]"
4,Translate the following words from Bengali to ...,মাটি,ਮਿੱਟੀ,"[238307, 75614]",soil,"[931, 485, 56389, 8360, 712, 62127, 174645, 23...",मिट्टी,"[118328, 212361, 4745, 235579]",মাটি,"[39808, 188842, 229052, 236821]",ਮਿੱਟੀ,"[238307, 75614]"


#### Block 9: Logit Lens Extraction Across All Layers for All 4 Candidates

Runs each prompt through the model, projects intermediate layer representations through `unemb`, and sums the probability mass over each candidate language's tokens.

In [ ]:
lang_probs = {c: [] for c in candidates}
entropy = []
energy = []
latents_all = []

for idx, d in tqdm(enumerate(dataset), total=len(dataset)):
    latents = llama.latents_all_layers(d['prompt'])
    logits = unemb(latents)
    last = logits[:, -1, :].float().softmax(dim=-1).detach().cpu()
    
    # Sum probabilities for each candidate language
    for c in candidates:
        tok_ids = torch.tensor(d[f'{c}_token_id'])
        lang_probs[c].append(last[:, tok_ids].sum(dim=-1))
        
    entropy += [compute_entropy(last)]
    latents_all += [latents[:, -1, :].float().detach().cpu().clone()]
    
    latents_normalized = latents[:, -1, :].float()
    latents_normalized = latents_normalized / (((latents_normalized**2).mean(dim=-1, keepdim=True))**0.5)
    latents_normalized /= (latents_normalized.norm(dim=-1, keepdim=True))
    norm = ((U_normalized @ latents_normalized.T)**2).mean(dim=0)**0.5
    energy += [norm / avgUU]

for c in candidates:
    lang_probs[c] = torch.stack(lang_probs[c])
    
entropy = torch.stack(entropy)
energy = torch.stack(energy)
latents = torch.stack(latents_all)

print("Latents shape:", latents.shape) 

 17%|███████████▎                                                     | 23/132 [07:06<33:04, 18.21s/it]

In [ ]:
# # Patch get_logits so input_ids are moved to the model's embedding device 
# embed_device = llama.model.model.embed_tokens.weight.device

# def get_logits_fixed(prompt):
#     inputs = llama.tokenizer(prompt, return_tensors="pt").to(embed_device)
#     with torch.no_grad():
#         logits = llama.model(inputs.input_ids).logits
#         return logits

# llama.get_logits = get_logits_fixed

In [ ]:
# Get the device of the input embedding layer
embed_device = llama.model.get_input_embeddings().weight.device

print("Embedding device:", embed_device)

def get_logits_fixed(prompt):
    inputs = llama.tokenizer(
        prompt,
        return_tensors="pt"
    ).to(embed_device)

    with torch.inference_mode():
        outputs = llama.model(**inputs)
        return outputs.logits

llama.get_logits = get_logits_fixed

In [ ]:
# Logit lens extraction with memory optimizations

# lang_probs = {c: [] for c in candidates}
# entropy = []
# energy = []
# latents_all = []

# complete_dataset = dataset
# # Taking a small part of dataset for testing other prompts
# # dataset = complete_dataset[:5]

# torch.cuda.empty_cache()

# with torch.no_grad():
#     for idx, d in tqdm(enumerate(dataset), total=len(dataset)):
#         # Extract all layers -> shape: [num_layers, seq_len, hidden_dim]
#         latents = llama.latents_all_layers(d['prompt'])
        
#         # Slice ONLY the last token BEFORE unembedding to save ~95% VRAM
#         last_latents = latents[:, -1, :] # shape: [num_layers, hidden_dim]
#         del latents
        
#         # Project only the last token through unemb
#         logits = unemb(last_latents.to(next(unemb.parameters()).device))
#         last = logits.float().softmax(dim=-1).detach().cpu()
#         del logits
        
#         # Sum probabilities for each candidate language
#         for c in candidates:
#             tok_ids = torch.tensor(d[f'{c}_token_id'])
#             lang_probs[c].append(last[:, tok_ids].sum(dim=-1))
            
#         entropy.append(compute_entropy(last))
        
#         # Move last_latents to CPU for storage and energy calculation
#         last_latents_cpu = last_latents.float().detach().cpu()
#         del last_latents
#         latents_all.append(last_latents_cpu.clone())
        
#         latents_normalized = last_latents_cpu / (((last_latents_cpu**2).mean(dim=-1, keepdim=True))**0.5)
#         latents_normalized /= (latents_normalized.norm(dim=-1, keepdim=True))
#         norm = ((U_normalized @ latents_normalized.T)**2).mean(dim=0)**0.5
#         energy.append(norm / avgUU)

# for c in candidates:
#     lang_probs[c] = torch.stack(lang_probs[c])
    
# entropy = torch.stack(entropy)
# energy = torch.stack(energy)
# latents = torch.stack(latents_all)

# print("Latents shape:", latents.shape)

#### Block 10: 4 Subplots (1 Per Candidate Language)

Generates a $2 \times 2$ grid where each subplot displays the layer-by-layer probability distribution (`plot_ci`) for one of the 4 candidate languages (`"en"`, `"hi"`, `input_lang`, `target_lang`), alongside the target language reference curve.

In [ ]:
# Single paper-style plot: candidate-language probabilities + target-language reference.
# Keep the original logit-lens calculations unchanged; only consolidate the plots.
size2tik = {'2b': 4, '7b': 5, '13b': 5, '70b': 10}
tik_step = size2tik.get(model_size, 4)
num_layers = entropy.shape[1]

# Use English as the primary candidate, then overlay Hindi and the target language.
primary_lang = 'en'
color_map = {
    'en': 'tab:orange',
    'hi': 'tab:green',
    target_lang: 'tab:blue',
    input_lang: 'tab:purple'
}

fig, ax, ax2 = plot_ci_plus_heatmap(
    lang_probs[primary_lang],
    entropy,
    primary_lang,
    color=color_map[primary_lang],
    tik_step=tik_step,
    do_colorbar=True,
    nums=[.99, 0.18, 0.025, 0.6]
)

# Overlay the other candidate-language curves on the same probability axis.
for c in ['hi', target_lang, input_lang]:
    if c != primary_lang:
        plot_ci(
            ax2,
            lang_probs[c],
            f'{c} (target)' if c == target_lang else c,
            color=color_map.get(c, 'tab:purple'),
            do_lines=False,
            tik_step=tik_step
        )

ax2.set_xlabel('layer')
ax2.set_ylabel('probability')
ax2.set_xlim(0, num_layers + 1)
ax2.set_ylim(0, 1)
ax2.legend(loc='upper left')

os.makedirs(save_path, exist_ok=True)
plt.savefig(
    f'{save_path}/{model_size}_{input_lang}_{target_lang}_probas_ent_single_plot.pdf',
    dpi=300,
    bbox_inches='tight'
)
plt.show()


#### Block 11: Energy Plot & Saving Latents

Plots the layer-wise energy curve and saves the extracted latent tensors.

In [ ]:
fig, ax2 = plt.subplots(figsize=(5, 3))
plot_ci(ax2, energy, 'energy', color='tab:green', do_lines=True, tik_step=tik_step)
ax2.set_xlabel('layer')
ax2.set_ylabel('energy')
ax2.set_xlim(0, num_layers + 1)

plt.savefig(f'{save_path}/{model_size}_{input_lang}_{target_lang}_energy.pdf', dpi=300, bbox_inches='tight')
plt.show()

torch.save(latents, f'{save_path}/{model_size}_{input_lang}_{target_lang}_latents.pt')
torch.save(lang_probs, f'{save_path}/{model_size}_{input_lang}_{target_lang}_lang_probs.pt')

In [ ]:
# ==============================================================================
# Cell: Translation Accuracy & Top-1 Decoded Output Inspection
# ==============================================================================

import pandas as pd
import torch

results = []
correct_exact_count = 0
correct_in_candidate_count = 0

print("Evaluating final-layer predictions across the dataset...\n")

with torch.no_grad():
    for idx, d in enumerate(dataset):
        prompt = d["prompt"]
        expected_target_str = d["out_token_str"].strip()
        expected_token_ids = set(d["out_token_id"])

        # 1. Get logits for the final prompt position
        # Using llama.get_logits (patched in Block 9 to avoid GPU device mismatches)
        logits = llama.get_logits(prompt)
        last_token_logits = logits[0, -1, :]  # Shape: [vocab_size]

        # 2. Top-1 greedy token predicted by the model
        top1_token_id = torch.argmax(last_token_logits).item()
        top1_token_str = (
            tokenizer.decode([top1_token_id])
            .strip()
            .replace("▁", "")
            .replace(" ", "")
        )

        # 3. Top-5 predictions for context
        top5_ids = torch.topk(last_token_logits, k=5).indices.tolist()
        top5_tokens = [
            tokenizer.decode([tid])
            .strip()
            .replace("▁", "")
            .replace(" ", "")
            for tid in top5_ids
        ]

        # 4. Accuracy checks
        # Check A: Did the model output one of the accepted token prefixes/ids?
        is_token_id_match = top1_token_id in expected_token_ids

        # Check B: Does the decoded string match or start the target word?
        is_exact_str_match = (top1_token_str == expected_target_str) or (
            expected_target_str.startswith(top1_token_str)
            and len(top1_token_str) > 0
        )

        is_correct = is_token_id_match or is_exact_str_match
        if is_correct:
            correct_exact_count += 1

        # Check C: Did it output the target word's English or Hindi equivalent instead?
        en_str = str(d.get("en_token_str", ""))
        hi_str = str(d.get("hi_token_str", ""))
        in_lang_str = str(d.get("in_token_str", ""))

        predicted_as = "Other"
        if is_correct:
            predicted_as = f"Target ({target_lang})"
        elif (
            top1_token_str.lower() == en_str.lower()
            or en_str.lower() in top1_token_str.lower()
        ):
            predicted_as = "English (en)"
        elif top1_token_str == hi_str or hi_str in top1_token_str:
            predicted_as = "Hindi (hi)"
        elif top1_token_str == in_lang_str or in_lang_str in top1_token_str:
            predicted_as = f"Input Lang ({input_lang})"

        # Probability assigned to the correct target tokens at layer 28
        target_prob_final = (
            lang_probs[target_lang][idx, -1].item()
            if target_lang in lang_probs
            else 0.0
        )

        results.append(
            {
                "Index": idx,
                "Input Word": in_lang_str,
                "Expected (Target)": expected_target_str,
                "Model Top-1 Output": top1_token_str,
                "Is Correct": "✅ Yes" if is_correct else "❌ No",
                "Predicted Language": predicted_as,
                "Target Prob (L28)": f"{target_prob_final:.4f}",
                "Top-5 Tokens": ", ".join(top5_tokens[:3]),
            }
        )

# Convert to DataFrame for display
eval_df = pd.DataFrame(results)

accuracy = (correct_exact_count / len(dataset)) * 100
print("=" * 60)
print(f"TRANSLATION TASK EVALUATION ({input_lang} -> {target_lang})")
print(f"Total Evaluated Samples : {len(dataset)}")
print(f"Correct Top-1 Predictions: {correct_exact_count} / {len(dataset)}")
print(f"Top-1 Greedy Accuracy   : {accuracy:.2f}%")
print("=" * 60)
print("\nBreakdown of what the model actually outputs:")
print(eval_df["Predicted Language"].value_counts())
print("\nSample Predictions:")
display(
    eval_df[
        [
            "Input Word",
            "Expected (Target)",
            "Model Top-1 Output",
            "Is Correct",
            "Predicted Language",
            "Target Prob (L28)",
            "Top-5 Tokens",
        ]
    ].head(15)
)

In [ ]:
# Inspect what the model is actually predicting instead of Tamil
print("Most common tokens predicted instead of the target language:")
print(eval_df['Model Top-1 Output'].value_counts().head(15))

print("\nSample predictions marked as 'Other':")
display(eval_df[eval_df['Predicted Language'] == 'Other'][['Input Word', 'Expected (Target)', 'Model Top-1 Output', 'Top-5 Tokens']].head(10))